# Validate REST-as-MCP PoC

Walks through the three-layer Proxy → Adapter → REST setup end-to-end against a running Tyk gateway with this branch's changes built in.

**Prereqs**
- Gateway built from `TT-17123-poc-api-to-mcp-v2` and running locally.
- A reachable upstream HTTP service the REST API can target. The notebook uses `https://httpbin.org` as the default upstream because it returns predictable echo responses.
- `pip install requests` in your notebook kernel.

Edit `GATEWAY_URL` and `GATEWAY_SECRET` below to match your gateway.

In [ ]:
import json, uuid, time, requests

GATEWAY_URL    = "http://localhost:8080"
GATEWAY_SECRET = "352d20ee67be67f6340b4c0605b044b7"  # X-Tyk-Authorization

ADMIN_HEADERS = {"X-Tyk-Authorization": GATEWAY_SECRET, "Content-Type": "application/json"}

# Stable IDs so cells can be re-run without orphans.
REST_API_ID    = "orders-rest"
MCP_PROXY_ID   = "orders-mcp-proxy"
ADAPTER_API_ID = f"{REST_API_ID}__mcp-server"

REST_LISTEN_PATH  = "/orders/"
PROXY_LISTEN_PATH = "/mcp/orders/"
REST_UPSTREAM     = "https://httpbin.org"

ORG_ID = "5e9d9544a1dcd60001d0ed20"  # any test org id; just keep it stable

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

def admin(method, path, **kw):
    return requests.request(method, f"{GATEWAY_URL}{path}", headers=ADMIN_HEADERS, timeout=15, **kw)

def reload_gateway():
    r = admin("GET", "/tyk/reload/group")
    assert r.status_code == 200, (r.status_code, r.text)
    # cold-start reloads are async; pause briefly for the loader to finish.
    time.sleep(1.5)

## 1. Health check

Confirms the gateway is reachable and the admin token is valid.

In [ ]:
r = admin("GET", "/hello")
print("status:", r.status_code)
print("body:", r.text[:200])
assert r.status_code == 200, "gateway not reachable — fix GATEWAY_URL / GATEWAY_SECRET first"

## 2. Create the REST API (Layer C)

OAS document with two operations and the new `server.mcp.enabled: true` marker. The gateway will synthesise the paired adapter on reload.

In [ ]:
rest_oas = {
    "openapi": "3.0.3",
    "info": {"title": "Orders", "version": "1.0.0"},
    "paths": {
        "/get": {
            "get": {
                "operationId": "echoGet",
                "summary": "Echo a GET request via httpbin",
                "parameters": [
                    {"name": "q", "in": "query", "schema": {"type": "string"}},
                ],
                "responses": {"200": {"description": "OK"}},
            }
        },
        "/post": {
            "post": {
                "operationId": "echoPost",
                "summary": "Echo a POST request via httpbin",
                "requestBody": {
                    "required": True,
                    "content": {"application/json": {"schema": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "qty":  {"type": "integer"},
                        },
                        "required": ["name"],
                    }}},
                },
                "responses": {"200": {"description": "OK"}},
            }
        },
    },
    "x-tyk-api-gateway": {
        "info": {
            "id":    REST_API_ID,
            "name":  "Orders REST",
            "orgId": ORG_ID,
            "state": {"active": True},
        },
        "server": {
            "listenPath": {"value": REST_LISTEN_PATH, "strip": True},
            "mcp": {"enabled": True, "curation": "expose-all"},
        },
        "upstream": {"url": REST_UPSTREAM},
    },
}

def upsert_oas(api_id, oas_doc):
    existing = admin("GET", f"/tyk/apis/oas/{api_id}")
    if existing.status_code == 200:
        r = admin("PUT", f"/tyk/apis/oas/{api_id}", data=json.dumps(oas_doc))
    else:
        r = admin("POST", "/tyk/apis/oas", data=json.dumps(oas_doc))
    print(f"{api_id}: {r.status_code} {r.text[:200]}")
    assert r.status_code in (200, 201), r.text
    return r

upsert_oas(REST_API_ID, rest_oas)
reload_gateway()

## 3. Verify the REST chain still serves REST traffic

REST clients see no change — the `mcp.enabled` marker does not interfere with normal traffic.

In [ ]:
r = requests.get(f"{GATEWAY_URL}{REST_LISTEN_PATH}get?q=hello", timeout=15)
print("status:", r.status_code)
print("body[:300]:", r.text[:300])

## 4. Create the MCP proxy (Layer A)

Operator-managed OAS APIDef posted to **`/tyk/mcps`** whose `upstream.url` is `tyk://<rest>__mcp-server`. `/tyk/mcps` is the single management endpoint for both classic remote-MCP proxies and REST-as-MCP proxies — when the upstream resolves to a synthetic adapter, the admit handler skips `MarkAsMCP()` so the proxy stays a plain reverse-proxy, letting the adapter own all JSON-RPC semantics. The admit-time validator (`validateMCP`) checks the REST API exists, is `mcp.enabled`, shares OrgID, and is not already paired.

In [ ]:
proxy_oas = {
    "openapi": "3.0.3",
    "info": {"title": "Orders MCP Proxy", "version": "1.0.0"},
    "paths": {},
    "x-tyk-api-gateway": {
        "info": {
            "id":    MCP_PROXY_ID,
            "name":  "Orders MCP Proxy",
            "orgId": ORG_ID,
            "state": {"active": True},
        },
        "server": {
            "listenPath":     {"value": PROXY_LISTEN_PATH, "strip": True},
            "authentication": {"enabled": False},
        },
        "upstream": {"url": f"tyk://{ADAPTER_API_ID}"},
    },
}

def upsert_mcp(api_id, oas_doc):
    existing = admin("GET", f"/tyk/mcps/{api_id}")
    if existing.status_code == 200:
        r = admin("PUT", f"/tyk/mcps/{api_id}", data=json.dumps(oas_doc))
    else:
        r = admin("POST", "/tyk/mcps", data=json.dumps(oas_doc))
    print(f"{api_id}: {r.status_code} {r.text[:300]}")
    assert r.status_code in (200, 201), r.text
    return r

# /tyk/mcps admit-time validation requires the source REST API to be
# already loaded into apisByID. Reload after step 2 must complete
# before this cell runs.
upsert_mcp(MCP_PROXY_ID, proxy_oas)
reload_gateway()

## 5. JSON-RPC: `initialize`

Answered inline by the adapter from its in-memory state. No upstream hop.

In [ ]:
def rpc(method, params=None, rid=None):
    body = {
        "jsonrpc": "2.0",
        "id":      rid or str(uuid.uuid4()),
        "method":  method,
    }
    if params is not None:
        body["params"] = params
    r = requests.post(
        f"{GATEWAY_URL}{PROXY_LISTEN_PATH}",
        headers={"Content-Type": "application/json"},
        data=json.dumps(body),
        timeout=20,
    )
    print("HTTP", r.status_code)
    try:
        return r.status_code, r.json()
    except Exception:
        return r.status_code, {"raw": r.text}

status, env = rpc("initialize", {"protocolVersion": "2025-06-18", "clientInfo": {"name": "notebook"}})
pp(env)
assert status == 200
assert env.get("result", {}).get("protocolVersion"), env

## 6. JSON-RPC: `tools/list`

Returns tools derived from the REST OAS by `oas.DeriveSourceTools` on the last reload — no persisted snapshot.

In [ ]:
status, env = rpc("tools/list")
pp(env)
tool_names = sorted(t["name"] for t in env["result"]["tools"])
print("tool names:", tool_names)
assert {"echoGet", "echoPost"}.issubset(tool_names), tool_names

## 7. JSON-RPC: `tools/call` (GET with query arg)

The adapter expands the `q` argument into the upstream query string, dispatches via `tyk://`, the REST chain runs (including `MCPLoopAuthBypass` which short-circuits auth for this paired-proxy traffic), and the upstream JSON is wrapped as MCP `result.content[0].text`.

In [ ]:
status, env = rpc("tools/call", {"name": "echoGet", "arguments": {"q": "hello-from-mcp"}})
pp(env)
result = env["result"]
assert result["isError"] is False, result
echoed = json.loads(result["content"][0]["text"])
assert echoed["args"].get("q") == "hello-from-mcp", echoed["args"]
print("✓ upstream observed our query arg via the REST chain")

## 8. JSON-RPC: `tools/call` (POST with body fields)

Body fields are expanded per `ParamLocations` (`body.<field>`) and posted as application/json.

In [ ]:
status, env = rpc("tools/call", {"name": "echoPost", "arguments": {"name": "widget", "qty": 7}})
pp(env)
result = env["result"]
assert result["isError"] is False, result
echoed = json.loads(result["content"][0]["text"])
assert echoed["json"] == {"name": "widget", "qty": 7}, echoed["json"]
print("✓ upstream observed the JSON body the adapter built")

## 9. Negative: forgery attempt

Hit the REST listen path directly. There is no paired-proxy trust descriptor on the request, so the REST API's normal auth path runs. With the REST API in this notebook left keyless for simplicity, this returns the upstream response — i.e. equivalent to a direct REST call. To exercise the actual forgery rejection, set `authentication.enabled: true` on the REST API and re-run.

In [ ]:
r = requests.get(f"{GATEWAY_URL}{REST_LISTEN_PATH}get?q=direct", timeout=15)
print("status:", r.status_code)
print("body[:300]:", r.text[:300])
# When the REST API is keyless, this succeeds. When auth is enabled on
# the REST API, this returns 401/403 — the bypass middleware does not
# fire because no trust descriptor was stamped by an adapter.

## 10. Negative: admit-time multi-tenant rejection

POST a second MCP proxy via `/tyk/mcps` from a different OrgID, pointing at the same adapter. `validateMCP` must reject with 4xx. We use `/tyk/mcps` here (rather than `/tyk/apis/oas`) because that path runs the admit-time validation hook this test is targeting.

In [ ]:
evil = {
    "openapi": "3.0.3",
    "info": {"title": "Evil Proxy", "version": "1.0.0"},
    "paths": {},
    "x-tyk-api-gateway": {
        "info": {
            "id":    "evil-mcp-proxy",
            "name":  "Evil MCP Proxy",
            "orgId": "some-other-org-aaaaaaaaaaaaaa",
            "state": {"active": True},
        },
        "server": {
            "listenPath":     {"value": "/evil-mcp/", "strip": True},
            "authentication": {"enabled": False},
        },
        "upstream": {"url": f"tyk://{ADAPTER_API_ID}"},
    },
}
r = admin("POST", "/tyk/mcps", data=json.dumps(evil))
print("status:", r.status_code)
print("body:", r.text[:300])
assert r.status_code >= 400, "expected validateMCP to reject cross-org pairing"
print("✓ admit-time multi-tenant check rejected the cross-org proxy")

## 11. Hot reload picks up REST OAS edits

Add a new operation to the REST OAS, reload, re-list tools. The proxy doesn't change. `shouldReloadSpec` is aware of MCP-exposed REST specs (and of synthetic adapters), so the paired adapter is rebuilt from fresh OAS on every reload.

In [ ]:
rest_oas["paths"]["/headers"] = {
    "get": {
        "operationId": "echoHeaders",
        "summary":     "Return the request headers httpbin saw",
        "responses":   {"200": {"description": "OK"}},
    }
}
upsert_oas(REST_API_ID, rest_oas)
reload_gateway()

status, env = rpc("tools/list")
names = sorted(t["name"] for t in env["result"]["tools"])
print("tool names after reload:", names)
assert "echoHeaders" in names, names
print("\u2713 derivation re-ran on reload \u2014 no proxy edit needed")

## 12. Cleanup

In [ ]:
for path in (f"/tyk/mcps/{MCP_PROXY_ID}", f"/tyk/apis/oas/{REST_API_ID}"):
    r = admin("DELETE", path)
    print(path, r.status_code, r.text[:200])
reload_gateway()
print("done")